# Suite XRX — The Cpp RegExp dialect

`Expressions.Cpp.RegExp` models the regular expressions `std::regex` reads (ISO/IEC 14882, [re.grammar] and
[re.synopt]), one dialect for its six grammars: ECMAScript's (ECMA-262, third edition, as [re.grammar] modifies it),
POSIX's basic and extended regular expressions, and those of awk, grep and egrep. A `regex` names its grammar among its
`syntax_option_type` flags; its kinds are the constructs the grammars share, and which grammar has a construct is
checked by validation. No subterm in C++ is one, a pattern being a string there: its holder does. Its kinds are
validated, bind nothing, cover C++'s three editions, print in each grammar, are refused by evaluation by name, and are
checked complete against its grammars kept as data.

In [ ]:
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.C import Expressions as CFamily, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.Cpp import Expressions as CppFamily, RegExp as CppRegExp, Types as CppTypes
from mbse.Expressions.C.Preprocessor import Expressions as PP
from mbse.Expressions.Cpp.Expressions import Expressions as CPP
from mbse.Expressions.Cpp.RegExp import Expressions as XR, Text as XRText
from mbse.Expressions.Cpp.Types import Expressions as XT
from mbse.Expressions.Framework import Evaluators as FE, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises

store = Builders.Store()  # the C family's dialects, Cpp RegExp's install first: the installs run in any order
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, CppRegExp.install, CFamily.install, CppFamily.install,
                CTypes.install, CppTypes.install, CPreprocessor.install]:
    store.transaction(install)
store.transaction(CppRegExp.define)  # its editions


def term(spec, dialect=XR.DIALECT):
    """The term value `spec` writes, the root of `dialect`'s holder, written in a transaction."""
    with store.transaction():
        family, own = dialect.path.split(".")
        return F.term(getattr(getattr(store.Expressions, family), own).Expression(lambda e: e.root(dialect.spec(spec))))


validate = lambda spec: XR.DIALECT.validate(term(spec))  # noqa: E731
text = lambda spec, grammar="ECMAScript": XRText.text(term(spec), grammar)  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
char = lambda c, spelling=None: lambda u: u.character(  # noqa: E731: a character, of its code point, and its spelling
    lambda r: r.value(ord(c)) if spelling is None else r.raw(spelling).value(ord(c)))
alt = lambda *elements: lambda u: u.alternative(lambda r: r.elements(list(elements)))  # noqa: E731
pattern = lambda *alternatives: lambda u: u.pattern(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
capture = lambda *alternatives: lambda u: u.capturinggroup(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
group = lambda *alternatives: lambda u: u.group(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
def repeat(element, least, most=None, greedy=None, spelling=None):
    """A quantifier: `element{least,most}`, lazy where not `greedy`, as `spelling` writes it where given."""
    def write(r):
        r = r.min(least).element(element)
        r = r if most is None else r.max(most)
        r = r if greedy is None else r.greedy(greedy)
        return r if spelling is None else r.raw(spelling)
    return lambda u: u.quantifier(write)


cls = lambda *elements, negate=False: lambda u: u.characterclass(lambda r: r.negate(negate).elements(list(elements)))  # noqa: E731
span = lambda low, high: lambda u: u.characterclassrange(lambda r: r.min(low).max(high))  # noqa: E731
chars = lambda kind, negate=False: lambda u: u.characterset(lambda r: r.setKind(kind).negate(negate))  # noqa: E731
anchor = lambda kind, negate=False: lambda u: u.assertion(lambda r: r.assertionKind(kind).negate(negate))  # noqa: E731
ahead = lambda *alternatives, negate=False: lambda u: u.assertion(  # noqa: E731: `(?=...)`, `(?!...)`
    lambda r: r.assertionKind("lookahead").negate(negate).alternatives(list(alternatives)))
backref = lambda n: lambda u: u.backreference(lambda r: r.ref(n))  # noqa: E731
named = lambda kind, name: lambda u: getattr(u, kind)(lambda r: r.name(name))  # noqa: E731: `[:alpha:]`, `[.a.]`, `[=a=]`
regex = lambda flags, spec: lambda u: u.regex(lambda r: r.flags(flags).pattern(spec))  # noqa: E731
patterns = lambda *specs: lambda u: u.patternlist(lambda r: r.patterns(list(specs)))  # noqa: E731
error = lambda source: lambda u: u.error(lambda r: r.text(source))  # noqa: E731
missing = lambda kind: lambda u: u.missing(lambda r: r.kind(kind))  # noqa: E731
abc = pattern(alt(char("a"), char("b")), alt(char("c")))  # `ab|c`

<a id="XRX-01"></a>
## XRX-01 · Its union and holder: its kinds and invalid syntax, a branch of the framework's terms, reading its family's dialects; a holder refining the framework's, round-tripped through JSON

In [ ]:
assert XR.TAGS == ("regex", "patternlist", "pattern", "alternative", "group", "capturinggroup", "quantifier", "characterclass",
                   "characterclassrange", "characterset", "character", "assertion", "backreference", "exclass",
                   "collatingelement", "equivalence") and XR.HELPERS == () and XR.SINCE == {}
assert list(XR.DIALECT.kinds()) == union(XR.DIALECT.union) == [*XR.TAGS, "error", "missing"]
assert XR.DIALECT.name() == "Cpp RegExp" and XR.DIALECT.tag == "cpp_regexp" and XR.DIALECT.union == "Expressions.Cpp.RegExp.Term"
assert "cpp_regexp" in union("Expressions.Framework.Term") and "cpp_regexp" in union("Expressions.Framework.Dialect")
assert XR.DIALECT.reads() == (CPP.DIALECT, XT.DIALECT, PP.DIALECT)  # its family's dialects
assert not any(tag in union(dialect.union) for dialect in XR.DIALECT.reads() for tag in XR.TAGS)  # no subterm is one
Holder = store.registered("Expressions.Cpp.RegExp.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(XR.DIALECT.union) and list(Holder.adjacencies) == ["abi"]
with store.transaction():
    holder = store.Expressions.Cpp.RegExp.Expression(lambda e: e.root(XR.DIALECT.spec(regex(["extended", "icase"], abc))))
assert Validators.Validate(store)(Holder, holder) == [] and XR.DIALECT.validate(holder) == []
assert F.dialect_of(F.term(holder)) == XR.DIALECT.union and F.kind_of(F.term(holder))[0] == "regex"
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
with raises(Exception, match="'binaryexpression' is not a member of a union value"):  # its root holds its own terms
    with store.transaction():
        store.Expressions.Cpp.RegExp.Expression(lambda e: e.root(XR.DIALECT.spec(CPP.binary("+", CPP.identifier("x"), 1))))
assert XR.DIALECT.validate(term(CPP.identifier("x"), CPP.DIALECT), {"x"}) == []  # a C++ term, as C++ reads it

<a id="XRX-02"></a>
## XRX-02 · Its kinds: their properties in order, each term property its Term; forms and make round-trip

In [ ]:
props = lambda name: list(store.registered(f"Expressions.Cpp.RegExp.{name}").properties)  # noqa: E731
assert props("Regex") == ["flags", "pattern"] and props("PatternList") == ["patterns"]
assert props("Pattern") == props("Group") == props("CapturingGroup") == ["alternatives"] and props("Alternative") == ["elements"]
assert props("Quantifier") == ["raw", "min", "max", "greedy", "element"] and props("CharacterClass") == ["negate", "elements"]
assert props("CharacterClassRange") == ["min", "max"] and props("CharacterSet") == ["setKind", "negate"]
assert props("Character") == ["raw", "value"] and props("Assertion") == ["assertionKind", "negate", "alternatives"]
assert props("Backreference") == ["ref"] and props("ExClass") == props("CollatingElement") == props("Equivalence") == ["name"]
prop = lambda name, member: store.registered(f"Expressions.Cpp.RegExp.{name}").properties[member].schema  # noqa: E731
Term = store.registered(XR.DIALECT.union)
assert prop("Regex", "pattern") is prop("Quantifier", "element") is prop("CharacterClassRange", "max") is Term
assert prop("Pattern", "alternatives").element is prop("Alternative", "elements").element is prop("Assertion", "alternatives").element is Term
made = F.form(term(regex(["basic"], pattern(alt(repeat(char("a"), 2, 3), backref(1))))))
assert made.kind == "regex" and dict(made.attributes)["flags"].values == ("basic",) and [F.kind_of(a)[0] for a in made.arguments] == ["pattern"]
quantified = F.form(term(repeat(cls(span(char("a"), char("z"))), 1, greedy=False)))
assert quantified.attributes == {"min": 1, "greedy": False} and [F.kind_of(a)[0] for a in quantified.arguments] == ["characterclass"]
for spec in [regex(["grep"], patterns(abc, pattern(alt(chars("any"))))), ahead(alt(anchor("word", True)), negate=True),
             cls(named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), chars("digit"), negate=True)]:
    assert F.same(term(XR.DIALECT.make(F.form(term(spec)))), term(spec))

<a id="XRX-03"></a>
## XRX-03 · Its checks: what each kind's terms may be, and its attributes' values

In [ ]:
for spec, problems in [
    (regex([], alt(char("a"))), ["a regex's pattern cannot hold an alternative"]),
    (patterns(alt(char("a"))), ["a patternlist's patterns cannot hold an alternative"]),
    (pattern(char("a")), ["a pattern's alternatives cannot hold a character"]),
    (alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]),
    (group(alt(named("exclass", "alpha"))), ["argument 0: an alternative's elements cannot hold an exclass"]),
    (capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]),
    (repeat(alt(), 0), ["a quantifier's element cannot hold an alternative"]),
    (cls(alt()), ["a characterclass's elements cannot hold an alternative"]),
    (span(chars("digit"), named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]),
    (ahead(char("a")), ["an assertion's alternatives cannot hold a character"]),
    (repeat(char("a"), -1), ["a quantifier's min cannot be negative, got -1"]),
    (repeat(char("a"), 3, 2), ["a quantifier's max cannot be less than its min, got 2 and 3"]),
    (repeat(anchor("word"), 0), ["a quantifier cannot repeat a word assertion"]),
    (repeat(chars("word"), 0), []),  # `\w*`: a word characterset, which a quantifier repeats
    (lambda u: u.quantifier(lambda r: r.element(char("a"))), ["a quantifier needs a min"]),
    (lambda u: u.quantifier(lambda r: r.min(1)), ["a quantifier needs an element"]),
    (chars("alpha"), ["CharacterSet.setKind cannot be 'alpha'"]),
    (chars("any", True), ["an any characterset cannot be negated"]),
    (lambda u: u.characterset(lambda r: r), ["a characterset needs a setKind"]),
    (lambda u: u.character(lambda r: r.value(0x110000)), ["a character's value must be a code point, got 1114112"]),
    (lambda u: u.character(lambda r: r.raw("a")), ["a character needs a value"]),
    (anchor("both"), ["Assertion.assertionKind cannot be 'both'"]),
    (anchor("start", True), ["a start assertion cannot be negated"]),
    (lambda u: u.assertion(lambda r: r.assertionKind("end").alternatives([alt()])), ["an end assertion holds no alternatives"]),
    (lambda u: u.assertion(lambda r: r.negate(True)), ["an assertion needs an assertionKind"]),
    (backref(0), ["a backreference's ref must be positive, got 0"]),
    (named("exclass", "a:b"), ["an exclass's name cannot hold ':'"]),
    (named("collatingelement", "a.="), ["a collatingelement's name cannot hold '.'"]),
    (named("equivalence", ""), ["an equivalence needs a name"]),
]:
    assert validate(spec) == problems, (validate(spec), problems)
for spec in [abc, alt(), cls(span(char("a"), named("collatingelement", "z"))), repeat(ahead(alt(char("a"))), 0, 1),
             cls(chars("word", True), char("-"), named("equivalence", "e"))]:
    assert validate(spec) == [], spec

<a id="XRX-04"></a>
## XRX-04 · Its grammars: a regex's flags, its one grammar element, ECMAScript by default; what each grammar has, checked

In [ ]:
assert XR.GRAMMARS == ("ECMAScript", "basic", "extended", "awk", "grep", "egrep") and XR.ECMASCRIPT == "ECMAScript"
assert XR.SYNTAX_OPTIONS == ("icase", "nosubs", "optimize", "collate", *XR.GRAMMARS, "multiline")
assert XR.grammar(("icase", "awk")) == "awk" and XR.grammar(("icase",)) == XR.grammar(()) == "ECMAScript"  # the default
unflagged = lambda u: u.regex(lambda r: r.pattern(abc))  # noqa: E731: a regex with no flags, ECMAScript's
assert validate(unflagged) == [] and XRText.ToText(term(unflagged)) == "ab|c"
assert validate(regex(["icase", "verbose", "basic"], abc)) == ["'verbose' is not a syntax_option_type element",
                                                             "alternation is not in the basic grammar"]
assert validate(regex(["basic", "extended"], abc)) == ["a regex has at most one grammar element, got basic, extended"]
assert validate(regex(["basic"], missing("pattern"))) == ["pattern: missing pattern"] and validate(lambda u: u.regex(lambda r: r.flags(["grep"]))) == [
    "a regex needs a pattern"]
assert XR.FEATURES["alternation"] == ("ECMAScript", "extended", "awk", "egrep") and XR.FEATURES["a group"] == ("ECMAScript",)
ecma = pattern(alt(group(alt(char("a"))), ahead(alt(char("b"))), anchor("word"), chars("digit"), chars("space"),
                   chars("word", True), repeat(char("c"), 0, greedy=False), backref(12)))
assert validate(regex([], ecma)) == validate(regex(["ECMAScript", "multiline"], ecma)) == []
assert validate(regex(["extended"], ecma)) == [
    "a group is not in the extended grammar", "a lookahead assertion is not in the extended grammar",
    "a word assertion is not in the extended grammar", "a digit characterset is not in the extended grammar",
    "a space characterset is not in the extended grammar", "a word characterset is not in the extended grammar",
    "a lazy quantifier is not in the extended grammar", "a backreference is not in the extended grammar",
    "a backreference past 9 is not in the extended grammar"]
assert validate(regex(["grep"], pattern(alt(cls(chars("digit")), backref(9))))) == ["a digit characterset is not in the grep grammar"]
anchored = pattern(alt(anchor("start"), capture(alt(anchor("start"), char("a"))), anchor("end")), alt(anchor("end"), char("b")))
assert validate(regex(["egrep"], anchored)) == validate(regex(["awk"], anchored)) == []
assert validate(regex(["basic"], anchored)) == ["alternation is not in the basic grammar",
                                                "an anchor within the expression is not in the basic grammar"]
assert validate(regex(["basic"], pattern(alt(anchor("start"), char("a"), anchor("end"))))) == []  # where a BRE begins and ends
repeated = pattern(alt(repeat(repeat(char("a"), 1), 0, 1), repeat(anchor("start"), 0)))
assert validate(regex(["extended"], repeated)) == []
assert validate(regex(["ECMAScript"], repeated)) == ["a repeated quantifier is not in the ECMAScript grammar",
                                                     "a repeated anchor is not in the ECMAScript grammar"]
assert validate(regex(["extended"], pattern(alt(lambda u: u.quantifier(lambda r: r.min(1)))))) == [
    "pattern: argument 0: argument 0: a quantifier needs an element"]
assert validate(regex(["grep"], patterns(pattern(alt(char("a"))), pattern(alt(anchor("start")))))) == []  # each line's own
assert validate(regex(["egrep"], patterns(abc))) == [] and validate(regex(["awk"], patterns(abc))) == [
    "a patternlist is not in the awk grammar"]
assert validate(regex(["basic"], pattern(alt(repeat(missing("atom"), 1), error("\\{"))))) == [  # invalid syntax: its own
    "pattern: argument 0: argument 0: element: missing atom", "pattern: argument 0: argument 1: invalid syntax '\\\\{'"]

<a id="XRX-05"></a>
## XRX-05 · What it binds: nothing; a capturing group numbered, a backreference resolving it by its number

In [ ]:
assert not any(F.is_binder(b.schema) for b in store.registered(XR.DIALECT.union).branches)
grouped = term(regex([], pattern(alt(capture(alt(char("a"))), backref(1)))))
assert SY.free(grouped, XR.DIALECT) == set() and XR.DIALECT.validate(grouped) == []
assert [F.kind_of(t)[0] for t in F.walk(grouped)] == ["regex", "pattern", "alternative", "capturinggroup", "alternative",
                                                      "character", "backreference"]

<a id="XRX-06"></a>
## XRX-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
assert validate(error("(?<")) == ["invalid syntax '(?<'"] and text(error("(?<")) == "(?<"
assert validate(missing("atom")) == ["missing atom"] and text(alt(char("a"), missing("atom"))) == "a"
assert validate(pattern(alt(char("a"), error("[")))) == ["argument 0: argument 1: invalid syntax '['"]
assert text(pattern(alt(char("a"), error("[z")))) == "a[z"
with raises(ValueError, match="Cpp RegExp error has no text to print"):
    text(lambda u: u.error(lambda r: r))
with raises(ValueError, match="Cpp idexpression is not a regular expression to print"):
    XRText.ToText(term(CPP.identifier("x"), CPP.DIALECT))

<a id="XRX-07"></a>
## XRX-07 · Its editions: C++'s three, each by its designation and common name, each with every kind

In [ ]:
regexp = store.Instances.Expressions.Cpp.RegExp
assert [FS.tags(s) for s in FS.standards(regexp).values()] == [
    ("ISO/IEC 14882:2017", "C++17"), ("ISO/IEC 14882:2020", "C++20"), ("ISO/IEC 14882:2024", "C++23")]
assert FS.properties(FS.find(regexp, "C++17"))["revision"] == ("year", 2017)
assert all([s.name for s in FS.kinds(e)] == [f"Expressions.Cpp.RegExp.{name}" for name in (
    "Regex", "PatternList", "Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier", "CharacterClass",
    "CharacterClassRange", "CharacterSet", "Character", "Assertion", "Backreference", "ExClass", "CollatingElement",
    "Equivalence")] for e in FS.standards(regexp).values())
assert store.transaction(CppRegExp.define)[0]["Cpp20"] is FS.find(regexp, "C++20")  # run again, it writes nothing

<a id="XRX-08"></a>
## XRX-08 · Printing: every kind in each grammar, its escapes, a spelling as written

In [ ]:
every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0), repeat(char("c"), 1),
                    repeat(char("d"), 0, 1), repeat(char("e"), 2, 2), repeat(char("f"), 2), repeat(char("g"), 2, 5),
                    cls(span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"),
                        named("equivalence", "e"), char("]"), negate=True), chars("any"), backref(1), anchor("end")))
for grammar, source in [
    ("ECMAScript", "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]\\]].\\1$"),
    ("basic", "^\\(a|\\.\\)b*c\\{1,\\}d\\{0,1\\}e\\{2\\}f\\{2,\\}g\\{2,5\\}[^a-z[:digit:][.ch.][=e=]]].\\1$"),
    ("grep", "^\\(a|\\.\\)b*c\\{1,\\}d\\{0,1\\}e\\{2\\}f\\{2,\\}g\\{2,5\\}[^a-z[:digit:][.ch.][=e=]]].\\1$"),
    ("extended", "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]]].\\1$"),
    ("awk", "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]]].\\1$"),
]:
    assert text(regex([grammar], every)) == source == text(every, grammar), (grammar, text(regex([grammar], every)))
ecma = pattern(alt(group(alt(char("a"))), ahead(alt(char("b"))), ahead(alt(char("c")), negate=True), anchor("word"),
                   anchor("word", True), chars("digit"), chars("space", True), chars("word"), repeat(char("x"), 0, greedy=False),
                   repeat(char("y"), 1, 3, greedy=True)),
               alt(char("\n"), char("\x01"), char("\x7f"), char("é"), cls(char("^"), char("-"), char("\\"), char("\t"), char("a"))))
assert XRText.ToText(term(ecma)) == "(?:a)(?=b)(?!c)\\b\\B\\d\\S\\wx*?y{1,3}|\\n\\x01\\x7fé[\\^\\-\\\\\\ta]"
assert text(regex(["egrep"], patterns(abc, pattern(alt(char("(")))))) == "ab|c\n\\("
assert text(regex(["awk"], pattern(alt(char("\n"), char("\x07"), char("\x08"), char("\x01"), char("\x7f"), char("/"), char("+"),
                                       cls(char("\t"), char("+")))))) == "\\n\\a\\b\\001\\177/\\+[\\t+]"
assert text(regex(["basic"], pattern(alt(char("+"), char("*"), char("\n"), cls(char("\\")))))) == "+\\*\n[\\]"
assert text(regex(["basic"], pattern(alt(repeat(char("a"), 1, spelling="\\{1,\\}"), char("x", "\\x"))))) == "a\\{1,\\}\\x"  # as written
assert text(repeat(char("a"), 0, spelling="{0,}")) == "a{0,}" and text(char("A", "\\x41")) == "\\x41"
assert XRText.ToText(term(regex(["icase"], abc))) == "ab|c"  # its flags are basic_regex's constructor's other argument

<a id="XRX-09"></a>
## XRX-09 · Evaluation: each kind refused by its dialect's name

In [ ]:
interpreter = FE.Interpreter(XR.DIALECT, {}, evaluated=frozenset())  # no kind evaluated, each refused by its dialect's name
for spec in [regex([], abc), abc, char("a"), cls(), backref(1)]:
    with raises(NotImplementedError, match=f"Cpp RegExp {F.kind_of(term(spec))[0]} is not evaluated"):
        interpreter(term(spec))
with raises(ValueError, match="Cpp RegExp error is not evaluated"):
    interpreter(term(error("(")))

<a id="XRX-10"></a>
## XRX-10 · Completeness: grammars/cpp-regex.json maps every production of its grammars, and every kind is mapped

In [ ]:
grammar = json.loads(Path("../../grammars/cpp-regex.json").read_text(encoding="utf-8"))
nodes = grammar["nodes"]
assert len(nodes) == 78 and len({(n["source"], n["node"]) for n in nodes}) == 78 and "78 productions and tokens" in grammar["checked"]
assert all(set(n) - {"note"} in ({"node", "source", "kinds"}, {"node", "source", "not"}) for n in nodes)
mapped = {kind for n in nodes for kind in n.get("kinds", ())}
assert mapped == {f"RegExp.{tag}" for tag in XR.TAGS}  # every kind, exactly: invalid syntax the framework's, which no grammar has
sources = [n["source"] for n in nodes]
assert sorted(set(sources), key=sources.index) == [
    "ISO/IEC 14882", "ECMA-262 3rd edition, 15.10.1, as [re.grammar] modifies it", "ISO/IEC/IEEE 9945, XBD 9.5",
    "ISO/IEC/IEEE 9945, XCU awk, Regular Expressions", "ISO/IEC/IEEE 9945, XCU grep"]
added = ["ClassAtomExClass", "ClassAtomCollatingElement", "ClassAtomEquivalence", "ClassName", "ClassNameCharacter"]
assert all(any(n["node"] == name for n in nodes) for name in added)  # [re.grammar]'s own productions